In [38]:
!pip install open_clip_torch --quiet

In [39]:
# CELL 2 — CONFIG  ← only block you need to edit

DATASET_INPUT_PATH = "/kaggle/input/datasets/mithilesh2303/preprocessed-deepfashion"   # <-- update this
CROPPED_META_CSV   = f"{DATASET_INPUT_PATH}/cropped_metadata.csv"
CROPPED_IMG_DIR    = f"{DATASET_INPUT_PATH}/cropped_images"

CLIP_MODEL_NAME    = "ViT-L-14"
CLIP_PRETRAINED    = "openai"

TOP_K_LIST         = [5, 10, 15]
BATCH_SIZE         = 256
SEEDS              = [32, 75, 78, ]   # replace with your team's roll numbers
DEVICE             = "cuda"

import os
OUTPUT_DIR = "/kaggle/working/results_condA"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config ready.")

Config ready.


In [40]:
# CELL 3 — IMPORTS

import random, json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import open_clip

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Torch: 2.10.0+cu128 | CUDA: True


In [41]:
# CELL 4 — LOAD METADATA

meta = pd.read_csv(CROPPED_META_CSV)
print("Total rows:", len(meta))
print(meta['split'].value_counts())
meta.head()

Total rows: 52712
split
train      25882
query      14218
gallery    12612
Name: count, dtype: int64


,cropped_image_path,item_id,split
0,/kaggle/working/cropped_images/02_1_front.jpg,id_00000002,train
1,/kaggle/working/cropped_images/02_2_side.jpg,id_00000002,train
2,/kaggle/working/cropped_images/02_4_full.jpg,id_00000002,train
3,/kaggle/working/cropped_images/02_7_additional...,id_00000002,train
4,/kaggle/working/cropped_images/02_1_front.jpg,id_00000003,train


In [42]:
# CELL 5 — FIX PATHS & DROP MISSING IMAGES

import glob

# Build lookup: "02_1_front.jpg" → full actual path
all_files = glob.glob(os.path.join(CROPPED_IMG_DIR, "*.jpg"))
suffix_lookup = {}
for f in all_files:
    suffix = "_".join(os.path.basename(f).split("_")[-3:])  # "02_1_front.jpg"
    suffix_lookup[suffix] = f

def resolve_path(p):
    if os.path.exists(p):
        return p
    basename = os.path.basename(p)   # "02_1_front.jpg"
    if basename in suffix_lookup:
        return suffix_lookup[basename]
    return p

meta['cropped_image_path'] = meta['cropped_image_path'].apply(resolve_path)
exists_mask = meta['cropped_image_path'].apply(os.path.exists)
print(f"Missing images: {(~exists_mask).sum()} / {len(meta)}")
meta = meta[exists_mask].reset_index(drop=True)

train_df   = meta[meta['split'] == 'train'].reset_index(drop=True)
query_df   = meta[meta['split'] == 'query'].reset_index(drop=True)
gallery_df = meta[meta['split'] == 'gallery'].reset_index(drop=True)
print(f"Train: {len(train_df)} | Query: {len(query_df)} | Gallery: {len(gallery_df)}")

Missing images: 0 / 52712
Train: 25882 | Query: 14218 | Gallery: 12612


In [43]:
# CELL 6 — DATASET CLASS

class FashionDataset(Dataset):
    def __init__(self, df, preprocess):
        self.df         = df.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['cropped_image_path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224))
        return self.preprocess(img), row['item_id']

print("Dataset class defined.")

Dataset class defined.


In [44]:
# CELL 7 — METRIC FUNCTIONS

def recall_at_k(query_ids, gallery_ids, sim_matrix, k):
    hits = 0
    for i, qid in enumerate(query_ids):
        top_k_ids = [gallery_ids[j] for j in np.argsort(-sim_matrix[i])[:k]]
        if qid in top_k_ids:
            hits += 1
    return hits / len(query_ids)


def ndcg_at_k(query_ids, gallery_ids, sim_matrix, k):
    scores = []
    for i, qid in enumerate(query_ids):
        top_k_idx = np.argsort(-sim_matrix[i])[:k]
        gains = [1.0 if gallery_ids[j] == qid else 0.0 for j in top_k_idx]
        dcg   = sum(g / np.log2(r + 2) for r, g in enumerate(gains))
        idcg  = 1.0 / np.log2(2)   # ideal: one relevant item at rank 1
        scores.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(scores))


def map_at_k(query_ids, gallery_ids, sim_matrix, k):
    ap_list = []
    for i, qid in enumerate(query_ids):
        top_k_idx = np.argsort(-sim_matrix[i])[:k]
        hits, prec_sum = 0, 0.0
        for rank, j in enumerate(top_k_idx, 1):
            if gallery_ids[j] == qid:
                hits     += 1
                prec_sum += hits / rank
        ap_list.append(prec_sum / max(hits, 1))
    return float(np.mean(ap_list))


def compute_all_metrics(query_ids, gallery_ids, sim_matrix, k_list):
    out = {}
    for k in k_list:
        out[f'Recall@{k}'] = recall_at_k(query_ids, gallery_ids, sim_matrix, k)
        out[f'NDCG@{k}']   = ndcg_at_k(  query_ids, gallery_ids, sim_matrix, k)
        out[f'mAP@{k}']    = map_at_k(   query_ids, gallery_ids, sim_matrix, k)
    return out

print("Metric functions defined.")

Metric functions defined.


In [45]:
# CELL 8 — LOAD FROZEN CLIP

model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED, device=DEVICE
)
model.eval()
for p in model.parameters():
    p.requires_grad = False

print(f"Loaded {CLIP_MODEL_NAME} ({CLIP_PRETRAINED}) — all parameters frozen.")

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Loaded ViT-L-14 (openai) — all parameters frozen.


In [46]:
# CELL 9 — EMBEDDING EXTRACTION

@torch.no_grad()
def extract_embeddings(df, desc="Extracting"):
    loader = DataLoader(
        FashionDataset(df, preprocess),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    )
    all_embs, all_ids = [], []
    for imgs, ids in tqdm(loader, desc=desc, leave=False):
        imgs = imgs.to(DEVICE)
        emb  = model.encode_image(imgs)
        emb  = F.normalize(emb.float(), dim=-1)
        all_embs.append(emb.cpu().numpy())
        all_ids.extend(list(ids))
    return np.vstack(all_embs).astype('float32'), all_ids

print("Extractor defined.")

Extractor defined.


In [47]:
print("Gallery rows:", len(gallery_df))
print("Query rows:", len(query_df))

# check a few paths actually exist
for path in gallery_df['cropped_image_path'].head(5):
    print(os.path.exists(path), path)

Gallery rows: 12612
Query rows: 14218
True /kaggle/input/datasets/mithilesh2303/preprocessed-deepfashion/cropped_images/img_WOMEN_Dresses_id_00001964_02_1_front.jpg
True /kaggle/input/datasets/mithilesh2303/preprocessed-deepfashion/cropped_images/img_WOMEN_Denim_id_00001248_02_3_back.jpg
True /kaggle/input/datasets/mithilesh2303/preprocessed-deepfashion/cropped_images/img_WOMEN_Sweaters_id_00002480_01_1_front.jpg
True /kaggle/input/datasets/mithilesh2303/preprocessed-deepfashion/cropped_images/img_WOMEN_Dresses_id_00004825_01_3_back.jpg
True /kaggle/input/datasets/mithilesh2303/preprocessed-deepfashion/cropped_images/img_WOMEN_Denim_id_00001248_02_3_back.jpg


In [48]:
# CELL 10 — RUN CONDITION A ACROSS ALL SEEDS
# Since the model is fully frozen and deterministic, results will be
# identical across seeds — but we report all 3 for consistent formatting.

print("=" * 55)
print("CONDITION A — Vision-only Frozen CLIP  (α = 1)")
print("=" * 55)

# Extract gallery embeddings once (same across seeds)
print("\nExtracting gallery embeddings ...")
g_embs, g_ids = extract_embeddings(gallery_df, desc="Gallery")

print("Extracting query embeddings ...")
q_embs, q_ids = extract_embeddings(query_df, desc="Query")

print(f"Query:   {q_embs.shape}")
print(f"Gallery: {g_embs.shape}")

# Cosine similarity (both already L2-normed → dot product = cosine)
sim_matrix = q_embs @ g_embs.T   # (Q, G)
print(f"Similarity matrix: {sim_matrix.shape}")

CONDITION A — Vision-only Frozen CLIP  (α = 1)

Extracting gallery embeddings ...


Extracting query embeddings ...


Query:   (14218, 768)
Gallery: (12612, 768)
Similarity matrix: (14218, 12612)


In [49]:
# CELL 11 — COMPUTE & DISPLAY METRICS

all_seed_results = []

for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    metrics = compute_all_metrics(q_ids, g_ids, sim_matrix, TOP_K_LIST)
    all_seed_results.append(metrics)
    print(f"Seed {seed}: ", {k: f"{v:.4f}" for k, v in metrics.items()})

# Aggregate
print("\n" + "─" * 55)
print("  CONDITION A — mean ± std across seeds")
print("─" * 55)
agg = {}
for k in all_seed_results[0]:
    vals = [r[k] for r in all_seed_results]
    agg[k] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}
    print(f"  {k:<12}  {agg[k]['mean']:.4f} ± {agg[k]['std']:.4f}")

Seed 42:  {'Recall@5': '0.0082', 'NDCG@5': '0.0057', 'mAP@5': '0.0034', 'Recall@10': '0.0132', 'NDCG@10': '0.0088', 'mAP@10': '0.0040', 'Recall@15': '0.0155', 'NDCG@15': '0.0104', 'mAP@15': '0.0041'}
Seed 43:  {'Recall@5': '0.0082', 'NDCG@5': '0.0057', 'mAP@5': '0.0034', 'Recall@10': '0.0132', 'NDCG@10': '0.0088', 'mAP@10': '0.0040', 'Recall@15': '0.0155', 'NDCG@15': '0.0104', 'mAP@15': '0.0041'}
Seed 44:  {'Recall@5': '0.0082', 'NDCG@5': '0.0057', 'mAP@5': '0.0034', 'Recall@10': '0.0132', 'NDCG@10': '0.0088', 'mAP@10': '0.0040', 'Recall@15': '0.0155', 'NDCG@15': '0.0104', 'mAP@15': '0.0041'}

───────────────────────────────────────────────────────
  CONDITION A — mean ± std across seeds
───────────────────────────────────────────────────────
  Recall@5      0.0082 ± 0.0000
  NDCG@5        0.0057 ± 0.0000
  mAP@5         0.0034 ± 0.0000
  Recall@10     0.0132 ± 0.0000
  NDCG@10       0.0088 ± 0.0000
  mAP@10        0.0040 ± 0.0000
  Recall@15     0.0155 ± 0.0000
  NDCG@15       0.0104 

In [50]:
# CELL 12 — SAVE RESULTS

with open(f"{OUTPUT_DIR}/condA_raw_results.json", 'w') as f:
    json.dump({'per_seed': all_seed_results, 'aggregate': agg}, f, indent=2)

rows = []
for seed, r in zip(SEEDS, all_seed_results):
    row = {'seed': seed, 'condition': 'A'}
    row.update(r)
    rows.append(row)

# summary row
summary_row = {'seed': 'mean±std', 'condition': 'A'}
for k, v in agg.items():
    summary_row[k] = f"{v['mean']:.4f}±{v['std']:.4f}"
rows.append(summary_row)

results_df = pd.DataFrame(rows)
results_df.to_csv(f"{OUTPUT_DIR}/condA_metrics.csv", index=False)
print("Saved to", OUTPUT_DIR)
results_df

Saved to /kaggle/working/results_condA


,seed,condition,Recall@5,NDCG@5,mAP@5,Recall@10,NDCG@10,mAP@10,Recall@15,NDCG@15,mAP@15
0,42,A,0.008229,0.005734,0.003448,0.013223,0.008771,0.004049,0.015473,0.010368,0.004098
1,43,A,0.008229,0.005734,0.003448,0.013223,0.008771,0.004049,0.015473,0.010368,0.004098
2,44,A,0.008229,0.005734,0.003448,0.013223,0.008771,0.004049,0.015473,0.010368,0.004098
3,mean±std,A,0.0082±0.0000,0.0057±0.0000,0.0034±0.0000,0.0132±0.0000,0.0088±0.0000,0.0040±0.0000,0.0155±0.0000,0.0104±0.0000,0.0041±0.0000
